# Stage 5: retrieval evaluation

Run after training with your local VS Code Python kernel. It finds the project, uses CIFAR-100 under `data/` or `/data`, and saves embeddings, metrics, and the qualitative grid in the project `results/` folder. Evaluate the held-out test split only for a finalized configuration.

In [ ]:
from pathlib import Path
import sys
import torch

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'src' / 'eval.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the project folder in VS Code so this notebook can find src/eval.py.')
sys.path.insert(0, str(ROOT))

data_roots = [ROOT / 'data', Path('/data'), Path('C:/data')]
DATA_ROOT = next((p for p in data_roots if (p / 'cifar-100-python' / 'train').is_file()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(f'Extract cifar-100-python under one of: {data_roots}')

OUTPUT_ROOT = ROOT
CHECKPOINT = OUTPUT_ROOT / 'runs' / 'baseline' / 'best.pt'
if not CHECKPOINT.is_file():
    raise FileNotFoundError('Could not find runs/baseline/best.pt. Run Stage 4 first.')
print('Project:', ROOT)
print('Dataset:', DATA_ROOT)
print('Checkpoint:', CHECKPOINT)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import os
import subprocess

command = [
    sys.executable, '-m', 'src.eval', '--checkpoint', str(CHECKPOINT),
    '--data-root', str(DATA_ROOT), '--split-dir', str(OUTPUT_ROOT / 'splits'),
    '--output-dir', str(OUTPUT_ROOT / 'results'), '--device', 'auto',
    '--seed', '42', '--num-workers', '2', '--no-download',
]
print(' '.join(command))
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
subprocess.run(command, cwd=ROOT, env=env, check=True)

In [ ]:
import json
from IPython.display import Image, display

metrics_path = OUTPUT_ROOT / 'results' / 'test_metrics.json'
metrics = json.loads(metrics_path.read_text())
print(json.dumps(metrics, indent=2))
display(Image(filename=str(OUTPUT_ROOT / 'results' / 'retrieval_grid.png')))

## Optional README table

The generator reads the JSON so README results are never typed by hand. Set `UPDATE_README = True` below to update the project README. Faiss is checked when `faiss-cpu` is installed; otherwise the report marks it unavailable.

In [ ]:
UPDATE_README = False
if UPDATE_README:
    subprocess.run([
        sys.executable, str(ROOT / 'scripts' / 'update_readme_metrics.py'),
        '--metrics', str(OUTPUT_ROOT / 'results' / 'test_metrics.json'),
        '--readme', str(ROOT / 'README.md'),
    ], cwd=ROOT, check=True)
    print('README retrieval table updated.')